In [1]:
import pandas as pd
import os
import time 
import soundfile as sf 
from faster_whisper import WhisperModel
from jiwer import wer , cer
import pynvml



audio_dir = "dataset/audio"
tsv_file = "dataset/transcripts/line_index_female.tsv"

transcript=pd.read_csv(tsv_file,sep='\t',header=None)

transcript.columns=['index','sentence']

transcript.head()


/home/buvanesh/Downloads/speech_model/whisperenv/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/buvanesh/Downloads/speech_model/whisperenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,index,sentence
0,taf_02345_00348037167,ஆஸ்த்ரேலியப் பெண்ணுக்கு முப்பத்தி மூன்று ஆண்டு...
1,taf_07049_00155837462,ஸ்ரீரங்கம் கோவிலில் வெடிகுண்டு மிரட்டல்
2,taf_09705_01218130267,உங்களுடைய உணவுக் கட்டுப்பாட்டைச் சொன்னால் மற்ற...
3,taf_03219_00712757493,ரஹானே மட்டும் ஆறுதலாக முப்பத்தி ஆறு ரன்கள் குவ...
4,taf_00008_01305524612,மனோரமாவிற்கு பூபதி என்ற மகன் உள்ளார்


In [2]:
dataset=[]

for _, row in transcript.iterrows():
    idx = row["index"]
    text = row["sentence"]

    audio_path = os.path.join(audio_dir, f"{idx}.wav")

    if os.path.exists(audio_path):
        dataset.append((audio_path, text))

print("Total samples:", len(dataset))
print("dataset:",dataset[:5])

Total samples: 2335
dataset: [('dataset/audio/taf_02345_00348037167.wav', 'ஆஸ்த்ரேலியப் பெண்ணுக்கு முப்பத்தி மூன்று ஆண்டுகளுக்குப் பின்னர் இந்தியா இழப்பீடு வழங்கியது'), ('dataset/audio/taf_07049_00155837462.wav', 'ஸ்ரீரங்கம் கோவிலில் வெடிகுண்டு மிரட்டல்'), ('dataset/audio/taf_09705_01218130267.wav', 'உங்களுடைய உணவுக் கட்டுப்பாட்டைச் சொன்னால் மற்றவர்களுக்கும் உதவியாக இருக்கும்'), ('dataset/audio/taf_03219_00712757493.wav', 'ரஹானே மட்டும் ஆறுதலாக முப்பத்தி ஆறு ரன்கள் குவித்தார்'), ('dataset/audio/taf_00008_01305524612.wav', 'மனோரமாவிற்கு பூபதி என்ற மகன் உள்ளார்')]


In [4]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained("vasista22/whisper-tamil-large-v2")

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    "vasista22/whisper-tamil-large-v2",
    torch_dtype=torch.float16
).to(device)

print("Model loaded on:", device)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Model loaded on: cuda


In [ ]:
import os
import time
import torch
import pandas as pd
import librosa
from tqdm import tqdm
from jiwer import wer, cer

references = []
predictions = []

latencies = []
audio_durations = []

transcript=transcript.head(100)  # first 100 audio files is enough for testing 

for _, row in tqdm(transcript.iterrows(), total=len(transcript)):

    idx = row["index"]
    ref = row["sentence"]

    audio_path = os.path.join(audio_dir, f"{idx}.wav")

    audio, sr = librosa.load(audio_path, sr=16000)
    duration = len(audio) / sr
    audio_durations.append(duration)

    inputs = processor(audio, sampling_rate=sr, return_tensors="pt")
    inputs = {k: v.to(device).half() for k, v in inputs.items()}

    start = time.time()

    with torch.no_grad():
        predicted_ids = model.generate(**inputs)

    latency = time.time() - start
    latencies.append(latency)

    pred = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    references.append(ref)
    predictions.append(pred)




In [12]:
references = [str(r) if r is not None else "" for r in references]
predictions = [str(p) if p is not None else "" for p in predictions]

wer_score=wer(references,predictions)
cer_score=cer(references,predictions)

wer_score,cer_score

(0.24039829302987198, 0.05391527599486521)

In [ ]:
# Average latency 
avg_lat=sum(latencies)/len(latencies)

rtf = [l/d for l,d in zip(latencies, audio_durations)]  # rea;l time factor calculation ( the model at exact speed as audio )
print("Average RTF:", sum(rtf)/len(rtf))
print("Average latnecy:",avg_lat)

Average RTF: 1.0122837760768457
Average latnecy: 6.291631405353546


In [ ]:
transcript['audio_durations']=audio_durations
transcript['latencies']=latencies
transcript.head()



,index,sentence,audio_durations,latencies
0,taf_02345_00348037167,ஆஸ்த்ரேலியப் பெண்ணுக்கு முப்பத்தி மூன்று ஆண்டு...,6.485375,5.809720
1,taf_07049_00155837462,ஸ்ரீரங்கம் கோவிலில் வெடிகுண்டு மிரட்டல்,4.010688,2.821802
2,taf_09705_01218130267,உங்களுடைய உணவுக் கட்டுப்பாட்டைச் சொன்னால் மற்ற...,7.082687,6.289041
3,taf_03219_00712757493,ரஹானே மட்டும் ஆறுதலாக முப்பத்தி ஆறு ரன்கள் குவ...,6.826688,4.805692
4,taf_00008_01305524612,மனோரமாவிற்கு பூபதி என்ற மகன் உள்ளார்,4.266687,3.405315


In [ ]:
from jiwer import process_words
measures=process_words(references,predictions)

insertions=measures.insertions
ref_words=sum(len(r.split()) for r in references)

hallucination_rate=insertions/ref_words

print("insertions:",insertions)
print("reference words:",ref_words)
print("hallucination_rate:",hallucination_rate)

insertions: 39
reference words: 703
hallucination_rate: 0.05547652916073969


In [ ]:
# GPU Memory and utilisation 

import torch 

allocated = torch.cuda.memory_allocated()/(1024*3)
reserved = torch.cuda.memory_reserved()/(1024*3)

print('Gpu memory allocated :',allocated)
print('gpu memory reserved :',reserved)